In [1]:
# get the data maniplulation library 
import pandas as pd
import numpy as np
import re
import math
import gc
import plotly.graph_objects as go
import plotly.express as px
gc.collect()

10

Study 1: A massive cross-cultural survey of 29,744 participants from 157 countries, in 50 languages, took part in the study.<br>
They focused their analyses on data from countries with at least 200 participants, leaving 26,684 participants from 51 countries. <br>
Participants were excluded if: 
1. their average response time was shorter than 2sec per item, or
2. they provided 11 or more consecutive identical answers to the emotion questions,
3. they had +11/20 missing values from the emotion questions <br>
This left data from 24,221 participants. <br>
Participants’ demographic information in each country is reported in Table S1 in the online supplemental materials. <br>
Presumably, from that, analyses were based on data from 51 countries, each with at least 200 participating individuals. <br>
It tested how 20 distinct emotions uniquely predicted different aspects of well-being during sustained collective stress. <br>
### We want to know: 
1. What were the 50 possible languages that participants were surveyed in? (I see no more that 40)
- norwegian
- serbian
- turkish
- persian
- vietnam
- malay
- estonia
- hungary
- arabic
- bahasa
- brazil portoguse
- catalan
- chinese
- croatian
- cantonese
- danish
- dari
- dutch
- farsi
- filipino
- finnish
- german
- nipali
- slovenian
- greek
- hebrew
- kazakh
- czech
- russian
- Ukraine
- pashto
- slovenian
- romanian
- swedish
- japanese?
2. What unique languages were the 51 countries surveyed in, excluding english?
3. Assuming everyone in a country responded in the same language, For two or more countries who responded in the same language, such as Mexico, Spain, should we attempt to inform the mexican responses with data from BILA mexican spanish-english dictionaries?
4. For each distinct language, what is the lexical ellaboration for each of the 20 emotions?
5. For each distinct language, is there any statistical significance in the distribution of the results of any particular emotion?
6. If yes, can we connect it to our lexical ellaboration of those emotions in that language in the bila dataset? 


7. Is the assumption I made true? Perhaps respondents were given the option of what language they want to take the survey in?
- if yes, this might complicate things, or simplify them.
    - complicate in that we might have Arabic respondant from Norway, for example.
    - simplify in that we might be able to extract that there were 1200 arabic languages responses across an irrelevant number of countries.






In [2]:
Covid51countries = pd.read_csv('/data/home/asher.katz/data/hypocognition/Covid51countries.csv')
Covid51countries

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,gender,ladder,employment,losejob,covidsymptom,language,datasource,ISO3,longstring,na_count
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,1.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,3,0
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,2.0,7.0,3,NaN,NaN,ZH-S,snowball,AUS,3,0
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,1.0,4.0,1,NaN,0.0,ZH-S,snowball,AUS,5,0
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,2.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,4,0
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,1.0,5.0,1,NaN,0.0,ZH-S,snowball,AUS,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,2.0,4.0,4,NaN,0.0,EN,snowball,KEN,4,0
24217,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,1.0,6.0,5,0.0,0.0,EN,snowball,KEN,2,0
24218,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0
24219,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0


In [54]:
Covid51countries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24221 entries, 0 to 24220
Data columns (total 75 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   country             24221 non-null  int64  
 1   countryname         24221 non-null  object 
 2   StartDate           24221 non-null  object 
 3   EndDate             24221 non-null  object 
 4   Status              23721 non-null  float64
 5   Progress            23721 non-null  float64
 6   Duration            23721 non-null  float64
 7   Finished            23721 non-null  float64
 8   RecordedDate        23721 non-null  object 
 9   home                24080 non-null  float64
 10  gathering           24078 non-null  float64
 11  distance            24115 non-null  float64
 12  hands               24108 non-null  float64
 13  help_covid          24186 non-null  float64
 14  donate_covid        24184 non-null  float64
 15  volunteer_covid     24173 non-null  float64
 16  help

In [55]:
info_Covid51countries = (
    pd.DataFrame({
        "column": Covid51countries.columns,
        "non_null_count": Covid51countries.notna().sum().values,
        "dtype": Covid51countries.dtypes.values
    })
)

info_Covid51countries


,column,non_null_count,dtype
0,country,24221,int64
1,countryname,24221,object
2,StartDate,24221,object
3,EndDate,24221,object
4,Status,23721,float64
...,...,...,...
70,language,24221,object
71,datasource,24221,object
72,ISO3,24221,object
73,longstring,24221,int64


In [56]:
Covid51countries['language'].value_counts()

language
EN       6180
ZH-S     2594
AR       1753
JA       1317
RU        961
ZH-TW     841
ES        792
NL        743
ID        709
UK        687
HR        542
ES-ES     491
IT        463
SV        446
ISL       400
EL        395
SR        387
VI        341
PT-BR     323
DE        318
FR        306
FA        294
HU        293
BG        291
DA        291
MALTI     285
TR        274
FI        269
KAT       246
CA        196
ZH-T      163
MS        159
HE        157
AFRI      116
MAR        58
HI         33
RO         26
PL         20
KO         19
PT          9
FILIP       7
CS          6
DARI        5
NO          4
ET          4
KAZ         4
ML          1
SL          1
SK          1
Name: count, dtype: int64

In [57]:
Covid51countries['language'].nunique()

49

In [3]:
country_response_count = pd.DataFrame(Covid51countries['country'].value_counts())
country_response_count

,count
country,
38,2009
89,1352
128,1348
112,1290
196,952
55,824
180,761
193,698
195,662


In [59]:
Covid51countries['country'].nunique()

51

In [4]:
languages_per_country = Covid51countries.groupby('country')['language'].value_counts().reset_index(name="count")
languages_per_country = languages_per_country.merge(
    country_response_count,
    on="country",
    how="left"
)
languages_per_country = languages_per_country.rename(columns={"count_x": "lang_responses", "count_y": "total_responses"})
languages_per_country["percent"] = languages_per_country["lang_responses"] / languages_per_country["total_responses"] * 100
languages_per_country


,country,language,lang_responses,total_responses,percent
0,10,ZH-S,181,378,47.883598
1,10,EN,156,378,41.269841
2,10,ID,12,378,3.174603
3,10,ZH-T,10,378,2.645503
4,10,VI,5,378,1.322751
...,...,...,...,...,...
496,202,ZH-S,2,338,0.591716
497,202,FR,1,338,0.295858
498,202,JA,1,338,0.295858
499,202,PL,1,338,0.295858


In [5]:
country_meta = (
    Covid51countries
    .groupby("country", as_index=False)
    .agg(
        countryname=("countryname", "first"),
        ISO3=("ISO3", "first")
    )
)
languages_per_country = languages_per_country.merge(
    country_meta,
    on="country",
    how="left"
)
languages_per_country

,country,language,lang_responses,total_responses,percent,countryname,ISO3
0,10,ZH-S,181,378,47.883598,Australia,AUS
1,10,EN,156,378,41.269841,Australia,AUS
2,10,ID,12,378,3.174603,Australia,AUS
3,10,ZH-T,10,378,2.645503,Australia,AUS
4,10,VI,5,378,1.322751,Australia,AUS
...,...,...,...,...,...,...,...
496,202,ZH-S,2,338,0.591716,Vietnam,VNM
497,202,FR,1,338,0.295858,Vietnam,VNM
498,202,JA,1,338,0.295858,Vietnam,VNM
499,202,PL,1,338,0.295858,Vietnam,VNM


In [6]:
lang_names = pd.read_csv('/data/home/asher.katz/hypocognition/reference/lang_names.csv')
languages_per_country = languages_per_country.merge(lang_names, left_on="language", right_on="Code", how="left")
languages_per_country = languages_per_country.drop(columns=["Code"])
languages_per_country

,country,language,lang_responses,total_responses,percent,countryname,ISO3,Language_Name
0,10,ZH-S,181,378,47.883598,Australia,AUS,Simplified Chinese
1,10,EN,156,378,41.269841,Australia,AUS,NaN
2,10,ID,12,378,3.174603,Australia,AUS,Indonesian
3,10,ZH-T,10,378,2.645503,Australia,AUS,Traditional Chinese
4,10,VI,5,378,1.322751,Australia,AUS,Vietnamese
...,...,...,...,...,...,...,...,...
496,202,ZH-S,2,338,0.591716,Vietnam,VNM,Simplified Chinese
497,202,FR,1,338,0.295858,Vietnam,VNM,French
498,202,JA,1,338,0.295858,Vietnam,VNM,Japanese
499,202,PL,1,338,0.295858,Vietnam,VNM,NaN


In [13]:
languages_per_country[languages_per_country['countryname']=="Spain"]

,country,language,lang_responses,total_responses,percent,countryname,ISO3,Language_Name
364,173,CA,187,418,44.736842,Spain,ESP,Catalan
365,173,ES-ES,105,418,25.119617,Spain,ESP,Spanish (Spain)
366,173,ZH-S,61,418,14.593301,Spain,ESP,Simplified Chinese
367,173,ES,21,418,5.023923,Spain,ESP,Spanish (General)
368,173,EN,9,418,2.153110,Spain,ESP,NaN
369,173,ISL,5,418,1.196172,Spain,ESP,Icelandic
370,173,DA,4,418,0.956938,Spain,ESP,Danish
371,173,AR,3,418,0.717703,Spain,ESP,Arabic
372,173,BG,3,418,0.717703,Spain,ESP,Bulgarian
373,173,NL,3,418,0.717703,Spain,ESP,Dutch


In [8]:
languages_per_country['countryname'].unique()

array(['Australia', 'Brazil', 'Bulgaria', 'Canada', 'Chile', 'China',
       'Colombia', 'Croatia', 'Curacao', 'Denmark', 'Egypt', 'Finland',
       'France', 'Georgia', 'Germany', 'Ghana', 'Greece',
       'Hong Kong (S.A.R)', 'Hungary', 'Iceland', 'India', 'Indonesia',
       'Iran', 'Ireland', 'Israel', 'Italy', 'Japan', 'Jordan',
       'Kazakhstan', 'Kenya', 'Malaysia', 'Malta', 'Mongolia',
       'Netherlands', 'New Zealand', 'Pakistan', 'Peru', 'Russia',
       'Serbia', 'Singapore', 'South Africa', 'Spain', 'Sweden', 'Syria',
       'Taiwan', 'Trinidad and Tobago', 'Turkey', 'Ukraine',
       'United Kingdom (UK)', 'United States of America (USA)', 'Vietnam'],
      dtype=object)

In [11]:
hk = languages_per_country[languages_per_country['countryname']=='Hong Kong (S.A.R)']

In [12]:
hk.to_csv('~/data/hypocognition/output_results/hk.csv')

### Already, at the top of languages_per_country we can see that more people took the survey in Chinese than English in Australia.
### We can't use English responses, we need to figure out what the "primary" language is, even if it's not the majority.

In [67]:
# lets try setting a percentage threshold and gradually lower it until we have 51 countries in the table
for i in range(100, 0, -1):
    df = languages_per_country[languages_per_country['percent']>=i]
    if df['country'].nunique() < 51:
        continue
    else:
        break
df =df.reset_index(drop=True)
df

,country,language,lang_responses,total_responses,percent,countryname,ISO3
0,10,ZH-S,181,378,47.883598,Australia,AUS
1,10,EN,156,378,41.269841,Australia,AUS
2,26,PT-BR,306,325,94.153846,Brazil,BRA
3,28,BG,269,276,97.463768,Bulgaria,BGR
4,34,EN,195,286,68.181818,Canada,CAN
5,37,ES,469,493,95.131846,Chile,CHL
6,38,ZH-S,1988,2009,98.954704,China,CHN
7,39,ES,122,237,51.476793,Colombia,COL
8,39,ES-ES,113,237,47.679325,Colombia,COL
9,45,HR,524,528,99.242424,Croatia,HRV


In [68]:
df['country'].nunique()

51

In [69]:
eng = df[df['language']=='EN'][['countryname', "percent"]]
eng

,countryname,percent
1,Australia,41.269841
4,Canada,68.181818
17,Ghana,99.695122
22,India,62.962963
25,Ireland,96.961326
31,Kenya,100.000000
33,Malta,75.968992
34,Mongolia,90.384615
36,New Zealand,57.251908
38,Pakistan,99.640934


In [ ]:
eng.to_csv('~/data/hypocognition/output_results/english_countires.csv')

### We want to get the responses from
* countries whose primary language is not english
* in the primary language
* even when it's not the majority used language

some of these countries probably don't have english as their primary language,

lets check

In [80]:
# here is a table gemini made for me
pl = pd.read_csv('~/hypocognition/reference/primary_languages.csv')
pl

,ISO3,Language Code
0,AUS,EN
1,MYS,MS
2,MLT,MALTI
3,MNG,RU
4,NLD,NL
5,NZL,EN
6,PAK,EN
7,PER,ES
8,RUS,RU
9,SRB,SR


In [84]:
# we want the rows of languages_per_country for all 51 countries
# where languages_per_country['ISO3'] == pl['Country Code (ISO3)']
# and where pl['Primary Language'] is in languages_per_country['language']
# for i in range(len(pl)):
#     country = pl.loc[i, 'Country Code (ISO3)']
#     primary = pl.loc[i, 'Primary Language']
#     if country in languages_per_country['ISO3']
#     if primary in languages_per_country['language']
#      == pl['Country Code (ISO3)'] # the rows of languages_per_country that are the same as the one in pl
# pl['Primary Language'] in languages_per_country['language']
languages_per_country.where( (languages_per_country['ISO3'] == pl['ISO3'].values.any()) & (languages_per_country['language'] == pl['Language Code'].values.any()))

,country,language,lang_responses,total_responses,percent,countryname,ISO3
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
496,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,NaN,NaN,NaN,NaN,NaN,NaN,NaN
498,NaN,NaN,NaN,NaN,NaN,NaN,NaN
499,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
matched = languages_per_country.merge(
    pl,
    how="inner",
    left_on=["ISO3", "language"],
    right_on=["ISO3", "Language Code"]
)
matched

,country,language,lang_responses,total_responses,percent,countryname,ISO3,Language Code
0,10,EN,156,378,41.269841,Australia,AUS,EN
1,26,PT-BR,306,325,94.153846,Brazil,BRA,PT-BR
2,28,BG,269,276,97.463768,Bulgaria,BGR,BG
3,34,EN,195,286,68.181818,Canada,CAN,EN
4,37,ES,469,493,95.131846,Chile,CHL,ES
5,38,ZH-S,1988,2009,98.954704,China,CHN,ZH-S
6,39,ES,122,237,51.476793,Colombia,COL,ES
7,45,HR,524,528,99.242424,Croatia,HRV,HR
8,47,NL,180,235,76.595745,Curacao,CUW,NL
9,50,DA,148,233,63.519313,Denmark,DNK,DA


In [89]:
reordered = matched[['countryname',	'country','ISO3',	'language',	'lang_responses',	'total_responses',	'percent']]
reordered.sort_values(by="percent").reset_index(drop=True)

,countryname,country,ISO3,language,lang_responses,total_responses,percent
0,Kazakhstan,91,KAZ,KAZ,4,291,1.374570
1,Mongolia,120,MNG,RU,13,208,6.250000
2,India,81,IND,HI,31,243,12.757202
3,Malta,112,MLT,MALTI,284,1290,22.015504
4,Spain,173,ESP,ES-ES,105,418,25.119617
5,South Africa,170,ZAF,AFRI,112,426,26.291080
6,Peru,143,PER,ES,128,357,35.854342
7,Netherlands,128,NLD,NL,524,1348,38.872404
8,Australia,10,AUS,EN,156,378,41.269841
9,Germany,68,DEU,DE,291,567,51.322751


In [90]:
reordered.to_csv('~/data/hypocognition/output_results/countries_by_primary_language.csv')

In [93]:
reordered[reordered['language']=='EN'][['countryname', "percent"]]

,countryname,percent
0,Australia,41.269841
3,Canada,68.181818
15,Ghana,99.695122
23,Ireland,96.961326
29,Kenya,100.000000
34,New Zealand,57.251908
35,Pakistan,99.640934
39,Singapore,89.903846
45,Trinidad and Tobago,99.700599
48,United Kingdom (UK),68.580060


In [103]:
languages_per_country[languages_per_country['countryname']== "Mongolia"]

,country,language,lang_responses,total_responses,percent,countryname,ISO3
281,120,EN,188,208,90.384615,Mongolia,MNG
282,120,RU,13,208,6.250000,Mongolia,MNG
283,120,KO,2,208,0.961538,Mongolia,MNG
284,120,TR,2,208,0.961538,Mongolia,MNG
285,120,CS,1,208,0.480769,Mongolia,MNG
286,120,DE,1,208,0.480769,Mongolia,MNG
287,120,JA,1,208,0.480769,Mongolia,MNG


In [96]:
languages_per_country['language'].unique()

array(['ZH-S', 'EN', 'ID', 'ZH-T', 'VI', 'DA', 'AR', 'ES', 'AFRI',
       'ES-ES', 'ET', 'FR', 'JA', 'SR', 'ZH-TW', 'PT-BR', 'PT', 'IT',
       'BG', 'RU', 'FA', 'UK', 'NL', 'DE', 'MS', 'TR', 'HR', 'ISL', 'NO',
       'RO', 'FI', 'KAT', 'SV', 'EL', 'CA', 'HU', 'SL', 'CS', 'PL', 'SK',
       'MAR', 'HI', 'HE', 'KO', 'KAZ', 'ML', 'MALTI', 'FILIP', 'DARI'],
      dtype=object)

In [92]:
reordered[reordered['language']=='EN'][['countryname', "percent"]].to_csv('~/data/hypocognition/output_results/countries_with_english_primary_language.csv')


In [104]:
Covid51countries.columns

Index(['country', 'countryname', 'StartDate', 'EndDate', 'Status', 'Progress',
       'Duration', 'Finished', 'RecordedDate', 'home', 'gathering', 'distance',
       'hands', 'help_covid', 'donate_covid', 'volunteer_covid', 'help',
       'donate', 'volunteer', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness',
       'ERQ1_rumination', 'ERQ2_reappraisal', 'ERQ3_suppression',
       'ERQ4_socialsharing', 'ERQ5_distraction', 'ERQ6_acceptance', 'support',
       'connected', 'phy_healthy', 'mentally_healthy', 'stressed', 'tired',
       'depressed', 'res_1', 'res_2', 'euda_1', 'euda_2', 'swl', 'sympathy',
       'concerned', 'overwhelmed', 'distressed', 'self_vulnerable',
       'country_vulnerable', 'age', 'education', 'gender', 'ladder',
       'employment', 'losejob', 'covidsymptom', 'languag

In [ ]:
Covid51countries[['country', 'countryname', '']]